In [1]:
#Setting Up Environment

import os
from dotenv import load_dotenv
import chromadb

from pypdf import PdfReader
from openai import OpenAI
from pathlib import Path

In [2]:
#Loading API Key

load_dotenv()

client=OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

print("OPENAI Client Initialized!!")

OPENAI Client Initialized!!


In [3]:
#Loading PDF and Markdown documents -- creating one fucntion that reads both files
def load_documents(folder_path):

    documents = []

    folder = Path(folder_path)

    for file_path in folder.iterdir():

        if file_path.suffix.lower() == ".md":

            with open(file_path, "r", encoding="utf-8") as file:
                text = file.read()

            documents.append({
                "text": text,
                "source": file_path.name,
                "page": 0
            })

        elif file_path.suffix.lower() == ".pdf":

            reader = PdfReader(file_path)

            for page_number, page in enumerate(reader.pages, start=1):

                text = page.extract_text() or ""

                if text.strip():

                    documents.append({
                        "text": text,
                        "source": file_path.name,
                        "page": page_number
                    })

    return documents

In [4]:
documents = load_documents("documents")

print("Total document sections:", len(documents))

for document in documents:
    print(document["source"], document["page"])

Total document sections: 4
ai_notes.md 0
sample_notes.pdf 1
sample_notes.pdf 2
sample_notes.pdf 3


In [5]:
#Split documents into Chunks
def chunk_documents(documents, chunk_size=500, overlap=50):

    chunks = []

    for document in documents:

        text = document["text"]

        start = 0
        chunk_number = 1

        while start < len(text):

            end = start + chunk_size

            chunk_text = text[start:end]

            if chunk_text.strip():

                chunks.append({
                    "text": chunk_text,
                    "source": document["source"],
                    "page": document["page"],
                    "chunk": chunk_number
                })

            start += chunk_size - overlap
            chunk_number += 1

    return chunks

In [6]:
chunks = chunk_documents(documents)

print("Total chunks:", len(chunks))

for chunk in chunks[:3]:
    print("\n", chunk)

Total chunks: 11

 {'text': '# Artificial Intelligence Notes\n\n## Machine Learning\nMachine learning allows computers to learn patterns\nfrom data without being explicitly programmed for\nevery individual task.\n\n## Deep Learning\nDeep learning is a subset of machine learning that\nuses neural networks with multiple layers.\n\n## Retrieval Augmented Generation\nRAG combines document retrieval with language\ngeneration. It retrieves relevant information from\nexternal documents before generating an answer.\n\n## Embeddings\nEmbeddings repr', 'source': 'ai_notes.md', 'page': 0, 'chunk': 1}

 {'text': 'nerating an answer.\n\n## Embeddings\nEmbeddings represent text as numerical vectors.\nThey help computers compare the meaning of text.', 'source': 'ai_notes.md', 'page': 0, 'chunk': 2}

 {'text': ' Introduction to Artificial Intelligence\nArtificial Intelligence (AI) refers to the ability of machines to perform tasks that typically require human\nintelligence. These tasks include learnin

In [7]:
#Generating Embeddings
def generate_embeddings(texts):

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )

    return [
        item.embedding
        for item in response.data
    ]

In [8]:
chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

embeddings = generate_embeddings(chunk_texts)

print("Embeddings generated:", len(embeddings))
print("Vector dimensions:", len(embeddings[0]))

Embeddings generated: 11
Vector dimensions: 1536


In [9]:
#Storing eveything in Chromadb
chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = chroma_client.get_or_create_collection(
    name="day21_documents"
)

In [10]:
# Preparing IDs and Metadata
ids = []
texts = []
metadatas = []

for i, chunk in enumerate(chunks):

    ids.append(f"chunk_{i}")

    texts.append(chunk["text"])

    metadatas.append({
        "source": chunk["source"],
        "page": chunk["page"],
        "chunk": chunk["chunk"]
    })

In [11]:
if chunks:

    collection.upsert(
        ids=ids,
        documents=texts,
        embeddings=embeddings,
        metadatas=metadatas
    )

print("Stored records:", collection.count())

Stored records: 11


In [12]:
#Building the retrieval function
def retrieve_documents(question, top_k=3):

    query_response = client.embeddings.create(
        model="text-embedding-3-small",
        input=question
    )

    query_embedding = query_response.data[0].embedding

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(top_k, collection.count())
    )

    return results

In [13]:
question = "What is retrieval augmented generation?"

results = retrieve_documents(question)

for i, document in enumerate(results["documents"][0]):

    metadata = results["metadatas"][0][i]

    print("\nRetrieved passage:")
    print(document)

    print("Source:", metadata["source"])
    print("Page:", metadata["page"])


Retrieved passage:
# Artificial Intelligence Notes

## Machine Learning
Machine learning allows computers to learn patterns
from data without being explicitly programmed for
every individual task.

## Deep Learning
Deep learning is a subset of machine learning that
uses neural networks with multiple layers.

## Retrieval Augmented Generation
RAG combines document retrieval with language
generation. It retrieves relevant information from
external documents before generating an answer.

## Embeddings
Embeddings repr
Source: ai_notes.md
Page: 0

Retrieved passage:
ses store embeddings and perform similarity searches efficiently.
Popular vector databases include:
• ChromaDB 
• Pinecone 
• Weaviate 
• Milvus 
• Qdrant 
These databases are commonly used in Retrieval-Augmented Generation (RAG) applications.
Retrieval-Augmented Generation
RAG combines information retrieval with large language models.
Workflow:
1. Load documents. 
2. Split documents into chunks. 
3. Convert chunks into embeddi

In [14]:
#Generating the answer using the retrieved passages
def generate_answer(question, results):

    context_parts = []

    for i, document in enumerate(results["documents"][0]):

        metadata = results["metadatas"][0][i]

        source = metadata["source"]
        page = metadata["page"]

        context_parts.append(
            f"Source: {source}, Page: {page}\n{document}"
        )

    context = "\n\n".join(context_parts)

    response = client.responses.create(
        model="gpt-4.1-mini",

        instructions="""
        You are a helpful document question-answering assistant.

        Answer using only the provided document context.

        If the answer is not present in the context,
        clearly say that the documents do not contain
        enough information.

        Do not invent facts or sources.

        Mention the source filename and page number
        when they are available.
        """,

        input=f"""
        Document context:

        {context}

        User question:

        {question}

        Answer:
        """
    )

    return response.output_text

In [17]:
#Combining everything to one function
def chat_with_documents(question):

    results = retrieve_documents(question, top_k=3)

    if not results["documents"] or not results["documents"][0]:
        return "No relevant documents were found."

    answer = generate_answer(question, results)

    return answer

In [18]:
question = "What are embeddings?"

answer = chat_with_documents(question)

print(answer)

Embeddings are numerical representations of text that transform words, sentences, or documents into vectors of floating-point numbers. They allow machines to compare the semantic meaning between texts. (Source: sample_notes.pdf, Page: 2)


In [19]:
question = "What are embeddings?"

answer = chat_with_documents(question)

print(answer)

Embeddings are numerical representations of text that transform words, sentences, or documents into vectors of floating-point numbers. They allow machines to compare the semantic meaning between texts by representing text as numerical vectors. (source: sample_notes.pdf, Page 2; ai_notes.md, Page 0)


In [20]:
questions = [
    "What is machine learning?",
    "What is deep learning?",
    "How does RAG work?",
    "What are embeddings?",
    "What is a vector database?"
]

for question in questions:

    print("\nQUESTION:", question)

    print("ANSWER:", chat_with_documents(question))

    print("-" * 50)


QUESTION: What is machine learning?
ANSWER: Machine Learning (ML) is a subset of Artificial Intelligence (AI) that enables systems to learn from data without being explicitly programmed. Instead of following fixed rules, ML models identify patterns in historical data and use those patterns to make predictions. (source: sample_notes.pdf, Page 1)
--------------------------------------------------

QUESTION: What is deep learning?
ANSWER: Deep learning is a specialized branch of machine learning that uses artificial neural networks with multiple layers. It has achieved remarkable success in image recognition, speech processing, and natural language understanding. (Source: sample_notes.pdf, Page 1)
--------------------------------------------------

QUESTION: How does RAG work?
ANSWER: Retrieval-Augmented Generation (RAG) works by combining information retrieval with large language models. The workflow involves the following steps:

1. Load documents.
2. Split documents into chunks.
3. Co